# Chapter 2 - Lab 1: Tool use - retrieving a customer's KPIs

In this lab we build an agent with the **tool use pattern**. The agent is asked to retrieve the
service-experience KPIs of a Platinum customer, Priya Sharma (`C1001`).

*Finance version*: a `get_company_fundamentals` tool returned Apple's profitability, valuation and liquidity ratios.
*Customer-care version*: a `get_customer_kpis` tool returns three KPI families for a customer:

* **Value KPIs** (how much the customer is worth): spend, orders, average order value, margin contribution.
* **Service-experience KPIs** (how well we serve them): CSAT, contacts per order, first contact resolution, resolution time.
* **Loyalty and risk KPIs** (will they stay?): repeat contacts, complaint rate, return rate, NPS.

We use the OpenAI Agents SDK with `gpt-4o-mini`. The data is mock CRM data, so only an OpenAI key is needed.

In [ ]:
%pip install -q openai-agents pandas

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

In [ ]:
import pandas as pd
from agents import Agent, Runner, function_tool

## Mock CRM data

In [ ]:
# Mock CRM / analytics warehouse: yearly KPIs per customer (2026 = year to date).
# In production this would be a SQL query against your customer-data platform.
CUSTOMER_KPIS = {
    "C1001": {  # Priya Sharma - Platinum, Pune, customer since 2019
        "name": "Priya Sharma", "tier": "Platinum",
        "value_kpis": {
            "Annual Spend (INR)":        {"2023": 52000, "2024": 61000, "2025": 88000, "2026": 74000},
            "Orders":                    {"2023": 6,     "2024": 7,     "2025": 9,     "2026": 6},
            "Average Order Value (INR)": {"2023": 8667,  "2024": 8714,  "2025": 9778,  "2026": 12333},
            "Margin Contribution (INR)": {"2023": 9360,  "2024": 10980, "2025": 15840, "2026": 13320},
        },
        "service_kpis": {
            "CSAT (1-5)":                {"2023": 4.7,  "2024": 4.8,  "2025": 4.6,  "2026": 2.9},
            "Contacts per Order":        {"2023": 0.17, "2024": 0.14, "2025": 0.22, "2026": 0.83},
            "First Contact Resolution":  {"2023": 0.85, "2024": 0.88, "2025": 0.80, "2026": 0.40},
            "Avg Resolution Hours":      {"2023": 18,   "2024": 16,   "2025": 22,   "2026": 96},
        },
        "loyalty_risk_kpis": {
            "Repeat Contact Rate":       {"2023": 0.05, "2024": 0.04, "2025": 0.08, "2026": 0.60},
            "Complaint Rate":            {"2023": 0.00, "2024": 0.00, "2025": 0.11, "2026": 0.67},
            "Return Rate":               {"2023": 0.00, "2024": 0.14, "2025": 0.00, "2026": 0.00},
            "NPS (0-10)":                {"2023": 9,    "2024": 10,   "2025": 9,    "2026": 3},
        },
    },
    "C1004": {  # Vikram Singh - Silver, Jaipur, customer since 2025
        "name": "Vikram Singh", "tier": "Silver",
        "value_kpis": {
            "Annual Spend (INR)":        {"2023": None, "2024": None, "2025": 41000, "2026": 55500},
            "Orders":                    {"2023": None, "2024": None, "2025": 3,     "2026": 4},
            "Average Order Value (INR)": {"2023": None, "2024": None, "2025": 13667, "2026": 13875},
            "Margin Contribution (INR)": {"2023": None, "2024": None, "2025": 1200,  "2026": -2400},
        },
        "service_kpis": {
            "CSAT (1-5)":                {"2023": None, "2024": None, "2025": 3.9,  "2026": 3.5},
            "Contacts per Order":        {"2023": None, "2024": None, "2025": 1.33, "2026": 1.75},
            "First Contact Resolution":  {"2023": None, "2024": None, "2025": 0.75, "2026": 0.71},
            "Avg Resolution Hours":      {"2023": None, "2024": None, "2025": 30,   "2026": 41},
        },
        "loyalty_risk_kpis": {
            "Repeat Contact Rate":       {"2023": None, "2024": None, "2025": 0.20, "2026": 0.29},
            "Complaint Rate":            {"2023": None, "2024": None, "2025": 0.67, "2026": 0.75},
            "Return Rate":               {"2023": None, "2024": None, "2025": 0.67, "2026": 0.75},
            "NPS (0-10)":                {"2023": None, "2024": None, "2025": 6,    "2026": 5},
        },
    },
}

## Specify the tool

In [ ]:
@function_tool
def get_customer_kpis(customer_id: str) -> dict:
    """Get value, service-experience and loyalty/risk KPIs for a NovaMart customer, by year."""
    record = CUSTOMER_KPIS.get(customer_id.strip().upper())
    if record is None:
        return {"error": f"Unknown customer {customer_id}"}
    return record

The tool returns a structured record instead of the whole CRM. This is a deliberate design choice:
**give the agent only the fields it needs.** Less data means lower cost, less confusion, and less
exposure of personal data.

## Specify the KPI retrieval agent

In [ ]:
MODEL = "gpt-4o-mini"
kpi_retrieval_agent = Agent(
    name="Customer KPI Retrieval Agent",
    model=MODEL,
    instructions="""
You are a Customer KPI Retrieval Agent for NovaMart's customer-experience team.

Tasks:
1) Call the KPI tool for the specified customer.
2) Output the requested set of KPIs.
""",
    tools=[get_customer_kpis],
)

## Run the agent

We ask for the **most recent year**. Watch which year the agent picks:

In [ ]:
results = await Runner.run(kpi_retrieval_agent, """Proceed step by step:
1- List 3 service-experience KPIs and their significance for a retailer.
2- Provide the values of these three KPIs for customer C1001 (Priya Sharma) for the most recent year.
""")
print(results.final_output)

The model may report **2024 or 2025** as "the most recent year": it infers "now" from its training
cutoff, not from the calendar. Same lesson as Chapter 1, now inside an agent.

In [ ]:
results = await Runner.run(kpi_retrieval_agent, """Proceed step by step:
1- List 3 service-experience KPIs and explain their significance for a retailer.
2- Provide the values of these three KPIs for customer C1001 (Priya Sharma) for the most recent
   year, knowing that we are in 2026 (2026 figures are year to date).
""")
print(results.final_output)

Now the agent reports 2026 and should notice the collapse: CSAT 4.6 -> 2.9, first-contact
resolution 80% -> 40%, average resolution time 22 h -> 96 h.

**Practical rule**: always pass today's date into an agent's instructions.

## Look at the raw data directly

In [ ]:
pd.DataFrame(CUSTOMER_KPIS["C1001"]["service_kpis"]).T

## Discussion
1. Priya's 2026 spend is still high but her service KPIs collapsed. What does that tell a CX manager?
2. Which three KPIs would you put on a churn early-warning dashboard?